# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object (avoid subscripting or iterating)
metadata_json = dataset.metadata.to_json()

print(f"Dataset Title: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")
print(f"Published: {metadata_json['datePublished']}")
print(f"License: {metadata_json['license']}")
print(f"Data Collection Timeframe: {metadata_json.get('dataCollectionTimeframe', 'N/A')}")
print(f"Personal Sensitive Information Fields: {metadata_json.get('personalSensitiveInformation', [])}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

The `mlcroissant` library exposes each dataset entity by its `@id`. We will enumerate available record sets and their fields to guide selection for analysis.

In [ ]:
# List record sets by @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")

for record_set in record_sets:
    rs_id = record_set['@id']
    rs_name = record_set.get('name')
    print(f"Record Set Name: {rs_name}\n  @id: {rs_id}")

    # List fields and their @ids
    fields = record_set.get('field', [])
    print(f"  Number of fields: {len(fields)}")
    for field in fields:
        field_id = field['@id']
        field_name = field.get('name')
        field_type = field.get('dataType')
        print(f"    Field: {field_name} (@id: {field_id}, type: {field_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Extract all record sets into Pandas DataFrames
record_set_ids = [record_set['@id'] for record_set in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

# Preview columns for the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns for record set {first_rs_id}:\n{dataframes[first_rs_id].columns.tolist()}")
    print(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate filtering, normalization, and grouping with explicit field `@id` variables.

In [ ]:
# EDA on one numeric field in the main record set
# Replace the below @id values with those found in your dataset overview
# Example: 'cr:recordSet/MHSurveyMain', 'cr:field/gad_7_score', 'cr:field/village'

main_record_set_id = record_set_ids[0]  # Select the first record set for demo
df = dataframes[main_record_set_id]

# Find a numeric field for demonstration
numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # pick the first numeric field
else:
    print("No numeric field found.")
    numeric_field_id = None

# Choose a group field (e.g., village, gender, etc.)
group_field_id = None
for candidate in ['cr:field/village', 'cr:field/gender', 'cr:field/level_of_education']:
    if candidate in df.columns:
        group_field_id = candidate
        break
if not group_field_id and len(df.columns) > 1:
    group_field_id = df.columns[1]

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For demonstration, we'll show a histogram of a numeric field and a bar chart grouped by a selected category field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot a histogram of the numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Plot a bar chart grouped by group_field_id
    if group_field_id and df[group_field_id].nunique() < 20:  # Limit to reasonable group count
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        plt.figure(figsize=(10,5))
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, extracting, and analyzing a FAIR² survey dataset with `mlcroissant`. Using Croissant `@id` references, we explored record sets, fields, and performed basic EDA including grouping, filtering, normalization, and visualization.

**Key findings:**
- The dataset provides self-reported demographic and mental health survey data from Kilifi, Kenya.
- Data can be programmatically loaded using `mlcroissant` and referenced using entity `@id`.
- Numeric fields can be filtered and normalized, and group-level comparisons made by key demographics.

Further exploration could include deeper analysis of mental health scores versus demographics, missing data handling per protocol, and AI-ready feature engineering.